# 03 – Modeling
Goal: split the data, build a preprocessing pipeline, and train baseline models.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data import load_data
from src.features import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET

df = load_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]
print(X.shape, y.shape)

(7043, 17) (7043,)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Churn rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

Train: (5634, 17)  Test: (1409, 17)
Churn rate — train: 0.265, test: 0.265


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

In [4]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train_transformed.shape)

preprocessor.get_feature_names_out()[:10]

(5634, 42)


array(['num__tenure', 'num__MonthlyCharges', 'num__SeniorCitizen',
       'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__PhoneService_No',
       'cat__PhoneService_Yes', 'cat__MultipleLines_No'], dtype=object)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]


def evaluate(model, name):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    return pd.Series({m: scores[f"test_{m}"].mean() for m in scoring}, name=name)

In [6]:
results = pd.DataFrame([
    evaluate(DummyClassifier(strategy="most_frequent"), "Dummy (always 'No')"),
    evaluate(LogisticRegression(max_iter=1000), "Logistic Regression"),
]).round(3)

results

/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/glitter/Glitter Cooking/customer-churn-model/venv/lib/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.


,accuracy,precision,recall,f1,roc_auc,average_precision
Dummy (always 'No'),0.735,0.000,0.000,0.000,0.500,0.265
Logistic Regression,0.801,0.653,0.538,0.589,0.845,0.660
